In [1]:

# ============================================================
# NOTEBOOK 09 — NSL-KDD PREPROCESSING
# CELL 01 — ENVIRONMENT AND SOURCE DATA DISCOVERY
# ============================================================

from google.colab import drive
from pathlib import Path
from collections import defaultdict
import os
import sys

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/MSc_Project")
MYDRIVE = Path("/content/drive/MyDrive")

EXPECTED_NAMES = {
    "kddtrain+.txt": "train",
    "kddtest+.txt": "test",
    "kddtrain+.csv": "train",
    "kddtest+.csv": "test",
}

SEARCH_ROOTS = [
    PROJECT_ROOT / "data",
    MYDRIVE,
]

EXCLUDED_DIRECTORIES = {
    "models",
    "results",
    ".git",
    ".ipynb_checkpoints",
    "__pycache__",
    ".trash",
}

found_files = {}

print("=" * 72)
print("CELL 01 — NSL-KDD SOURCE DATA DISCOVERY")
print("=" * 72)
print("Python:", sys.version.split()[0])
print("Project root:", PROJECT_ROOT)

for search_root in SEARCH_ROOTS:

    if not search_root.is_dir():
        continue

    for current, dirs, files in os.walk(search_root):

        current_path = Path(current)

        depth = len(
            current_path.relative_to(search_root).parts
        )

        dirs[:] = [
            name for name in dirs
            if name.lower() not in EXCLUDED_DIRECTORIES
        ]

        if depth >= 6:
            dirs[:] = []

        for filename in files:

            if filename.lower() not in EXPECTED_NAMES:
                continue

            candidate = current_path / filename
            found_files[str(candidate)] = candidate

# Group source files by their parent directory

groups = defaultdict(dict)

for path in found_files.values():

    role = EXPECTED_NAMES[path.name.lower()]
    extension = path.suffix.lower()

    groups[(str(path.parent), extension)][role] = path

complete_pairs = [
    files for files in groups.values()
    if "train" in files and "test" in files
]

print("\nSOURCE FILE CANDIDATES")

if found_files:
    for path in sorted(found_files.values()):
        print(path)
else:
    print("No matching source files found.")

print("\nCOMPLETE DATASET PAIRS")
print("Pairs found:", len(complete_pairs))

for index, pair in enumerate(complete_pairs, 1):
    print(f"\nPair {index}")
    print("Training:", pair["train"])
    print("Testing :", pair["test"])

NSL_SOURCE_READY = len(complete_pairs) == 1
NSL_TRAIN_SOURCE = None
NSL_TEST_SOURCE = None

if NSL_SOURCE_READY:

    NSL_TRAIN_SOURCE = complete_pairs[0]["train"]
    NSL_TEST_SOURCE = complete_pairs[0]["test"]

    print("\nSELECTED SOURCES")
    print("Training:", NSL_TRAIN_SOURCE)
    print("Testing :", NSL_TEST_SOURCE)

    print(
        "Training size (MB):",
        round(NSL_TRAIN_SOURCE.stat().st_size / 1e6, 2)
    )
    print(
        "Testing size (MB):",
        round(NSL_TEST_SOURCE.stat().st_size / 1e6, 2)
    )

    print("\nCELL 01: PASSED")

else:
    print("\nCELL 01: SOURCE REVIEW REQUIRED")
    print(
        "Exactly one complete full-dataset pair "
        "must be identified before loading."
    )

print("Source files modified: NO")
print("=" * 72)


Mounted at /content/drive
CELL 01 — NSL-KDD SOURCE DATA DISCOVERY
Python: 3.12.13
Project root: /content/drive/MyDrive/MSc_Project

SOURCE FILE CANDIDATES
/content/drive/MyDrive/MSc_Project/data/NSL_KDD/raw/KDDTest+.txt
/content/drive/MyDrive/MSc_Project/data/NSL_KDD/raw/KDDTrain+.txt

COMPLETE DATASET PAIRS
Pairs found: 1

Pair 1
Training: /content/drive/MyDrive/MSc_Project/data/NSL_KDD/raw/KDDTrain+.txt
Testing : /content/drive/MyDrive/MSc_Project/data/NSL_KDD/raw/KDDTest+.txt

SELECTED SOURCES
Training: /content/drive/MyDrive/MSc_Project/data/NSL_KDD/raw/KDDTrain+.txt
Testing : /content/drive/MyDrive/MSc_Project/data/NSL_KDD/raw/KDDTest+.txt
Training size (MB): 19.11
Testing size (MB): 3.44

CELL 01: PASSED
Source files modified: NO


In [2]:

# ============================================================
# NOTEBOOK 09 — NSL-KDD PREPROCESSING
# CELL 02 — RAW DATASET AND SCHEMA VERIFICATION
# ============================================================

import pandas as pd
import numpy as np

if not globals().get("NSL_SOURCE_READY", False):
    raise RuntimeError("Complete Cell 01 first.")

NSL_FEATURE_COLUMNS = [
    "duration",
    "protocol_type",
    "service",
    "flag",
    "src_bytes",
    "dst_bytes",
    "land",
    "wrong_fragment",
    "urgent",
    "hot",
    "num_failed_logins",
    "logged_in",
    "num_compromised",
    "root_shell",
    "su_attempted",
    "num_root",
    "num_file_creations",
    "num_shells",
    "num_access_files",
    "num_outbound_cmds",
    "is_host_login",
    "is_guest_login",
    "count",
    "srv_count",
    "serror_rate",
    "srv_serror_rate",
    "rerror_rate",
    "srv_rerror_rate",
    "same_srv_rate",
    "diff_srv_rate",
    "srv_diff_host_rate",
    "dst_host_count",
    "dst_host_srv_count",
    "dst_host_same_srv_rate",
    "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate",
    "dst_host_serror_rate",
    "dst_host_srv_serror_rate",
    "dst_host_rerror_rate",
    "dst_host_srv_rerror_rate",
]

NSL_ALL_COLUMNS = NSL_FEATURE_COLUMNS + [
    "attack",
    "difficulty"
]

NSL_CATEGORICAL_COLUMNS = [
    "protocol_type",
    "service",
    "flag"
]

NSL_NUMERIC_COLUMNS = [
    column for column in NSL_FEATURE_COLUMNS
    if column not in NSL_CATEGORICAL_COLUMNS
]

print("=" * 72)
print("CELL 02 — RAW NSL-KDD DATASET VERIFICATION")
print("=" * 72)

print("Defined features:", len(NSL_FEATURE_COLUMNS))
print("Categorical features:", len(NSL_CATEGORICAL_COLUMNS))
print("Numeric features:", len(NSL_NUMERIC_COLUMNS))
print("Total raw columns:", len(NSL_ALL_COLUMNS))

assert len(NSL_FEATURE_COLUMNS) == 41
assert len(NSL_ALL_COLUMNS) == 43
assert len(set(NSL_ALL_COLUMNS)) == 43

NSL_RAW_TRAIN = pd.read_csv(
    NSL_TRAIN_SOURCE,
    header=None,
    names=NSL_ALL_COLUMNS,
    skipinitialspace=True,
    low_memory=False
)

NSL_RAW_TEST = pd.read_csv(
    NSL_TEST_SOURCE,
    header=None,
    names=NSL_ALL_COLUMNS,
    skipinitialspace=True,
    low_memory=False
)

expected_shapes = {
    "Training": (125973, 43),
    "Testing": (22544, 43)
}

for name, dataset in {
    "Training": NSL_RAW_TRAIN,
    "Testing": NSL_RAW_TEST
}.items():

    print("\n" + "-" * 72)
    print(name.upper())
    print("-" * 72)

    print("Shape:", dataset.shape)
    print("Missing values:", int(dataset.isna().sum().sum()))
    print("Attack labels:", dataset["attack"].nunique())
    print("Difficulty values:", dataset["difficulty"].nunique())

    if dataset.shape != expected_shapes[name]:
        raise RuntimeError(
            f"{name}: unexpected dataset shape {dataset.shape}."
        )

    if dataset.isna().any().any():
        raise RuntimeError(
            f"{name}: missing values detected."
        )

    difficulty = pd.to_numeric(
        dataset["difficulty"],
        errors="coerce"
    )

    if difficulty.isna().any():
        raise RuntimeError(
            f"{name}: invalid difficulty values."
        )

    for column in NSL_NUMERIC_COLUMNS:

        numeric_values = pd.to_numeric(
            dataset[column],
            errors="coerce"
        )

        if numeric_values.isna().any():
            raise RuntimeError(
                f"{name}: invalid numeric feature {column}."
            )

        if not np.isfinite(
            numeric_values.to_numpy(dtype=np.float64)
        ).all():
            raise RuntimeError(
                f"{name}: non-finite feature {column}."
            )

    print("Numeric feature validation: PASS")
    print("Difficulty column validation: PASS")

print("\nFEATURE STRUCTURE")

print("Categorical columns:", NSL_CATEGORICAL_COLUMNS)
print("Numeric columns:", len(NSL_NUMERIC_COLUMNS))
print("Attack target:", "attack")
print("Excluded auxiliary field:", "difficulty")

print("\nCELL 02: PASSED")
print("Original NSL partitions preserved: YES")
print("Records removed: NO")
print("Model training performed: NO")
print("=" * 72)


CELL 02 — RAW NSL-KDD DATASET VERIFICATION
Defined features: 41
Categorical features: 3
Numeric features: 38
Total raw columns: 43

------------------------------------------------------------------------
TRAINING
------------------------------------------------------------------------
Shape: (125973, 43)
Missing values: 0
Attack labels: 23
Difficulty values: 22
Numeric feature validation: PASS
Difficulty column validation: PASS

------------------------------------------------------------------------
TESTING
------------------------------------------------------------------------
Shape: (22544, 43)
Missing values: 0
Attack labels: 38
Difficulty values: 22
Numeric feature validation: PASS
Difficulty column validation: PASS

FEATURE STRUCTURE
Categorical columns: ['protocol_type', 'service', 'flag']
Numeric columns: 38
Attack target: attack
Excluded auxiliary field: difficulty

CELL 02: PASSED
Original NSL partitions preserved: YES
Records removed: NO
Model training performed: NO


In [3]:

# ============================================================
# NOTEBOOK 09 — NSL-KDD PREPROCESSING
# CELL 03 — ATTACK LABEL INVENTORY
# ============================================================

import pandas as pd

if (
    "NSL_RAW_TRAIN" not in globals()
    or "NSL_RAW_TEST" not in globals()
):
    raise RuntimeError("Complete Cell 02 first.")

print("=" * 72)
print("CELL 03 — ATTACK LABEL INVENTORY")
print("=" * 72)

def normalise_attack_labels(values):
    return (
        values.astype(str)
        .str.strip()
        .str.lower()
        .str.rstrip(".")
    )

NSL_TRAIN_ATTACKS = normalise_attack_labels(
    NSL_RAW_TRAIN["attack"]
)

NSL_TEST_ATTACKS = normalise_attack_labels(
    NSL_RAW_TEST["attack"]
)

if (
    NSL_TRAIN_ATTACKS.eq("").any()
    or NSL_TEST_ATTACKS.eq("").any()
):
    raise RuntimeError("Empty attack labels detected.")

train_counts = NSL_TRAIN_ATTACKS.value_counts()
test_counts = NSL_TEST_ATTACKS.value_counts()

all_attacks = sorted(
    set(train_counts.index) | set(test_counts.index)
)

test_only_attacks = sorted(
    set(test_counts.index) - set(train_counts.index)
)

train_only_attacks = sorted(
    set(train_counts.index) - set(test_counts.index)
)

inventory = pd.DataFrame({
    "Attack": all_attacks,
    "Training Count": [
        int(train_counts.get(name, 0))
        for name in all_attacks
    ],
    "Testing Count": [
        int(test_counts.get(name, 0))
        for name in all_attacks
    ]
})

inventory["Test-Only"] = (
    (inventory["Training Count"] == 0)
    & (inventory["Testing Count"] > 0)
)

print("\nDATASET LABEL SUMMARY")

print("Training records:", len(NSL_TRAIN_ATTACKS))
print("Testing records:", len(NSL_TEST_ATTACKS))
print("Unique training labels:", len(train_counts))
print("Unique testing labels:", len(test_counts))
print("Unique combined labels:", len(all_attacks))

print("\nATTACK LABEL INVENTORY")
print(inventory.to_string(index=False))

print("\nTEST-ONLY ATTACK NAMES")
print(test_only_attacks)

print("\nTRAIN-ONLY ATTACK NAMES")
print(train_only_attacks)

if "normal" not in all_attacks:
    raise RuntimeError("Normal class not found.")

if inventory["Training Count"].sum() != 125973:
    raise RuntimeError("Training label totals differ.")

if inventory["Testing Count"].sum() != 22544:
    raise RuntimeError("Testing label totals differ.")

NSL_ATTACK_INVENTORY_VERIFIED = True

print("\nINTEGRITY VERIFICATION")
print("Training label total: PASS")
print("Testing label total: PASS")
print("Normal class present: PASS")
print("CELL 03: PASSED")
print("Five-class mapping: PENDING")
print("=" * 72)


CELL 03 — ATTACK LABEL INVENTORY

DATASET LABEL SUMMARY
Training records: 125973
Testing records: 22544
Unique training labels: 23
Unique testing labels: 38
Unique combined labels: 40

ATTACK LABEL INVENTORY
         Attack  Training Count  Testing Count  Test-Only
        apache2               0            737       True
           back             956            359      False
buffer_overflow              30             20      False
      ftp_write               8              3      False
   guess_passwd              53           1231      False
     httptunnel               0            133       True
           imap              11              1      False
        ipsweep            3599            141      False
           land              18              7      False
     loadmodule               9              2      False
       mailbomb               0            293       True
          mscan               0            996       True
       multihop               7       

In [4]:

# ============================================================
# NOTEBOOK 09 — NSL-KDD PREPROCESSING
# CELL 04 — FIVE-CLASS ATTACK MAPPING
# ============================================================

import pandas as pd
from collections import Counter

print("=" * 72)
print("CELL 04 — FIVE-CLASS ATTACK MAPPING")
print("=" * 72)

required = [
    "NSL_RAW_TRAIN",
    "NSL_RAW_TEST",
    "NSL_TRAIN_ATTACKS",
    "NSL_TEST_ATTACKS"
]

missing = [name for name in required if name not in globals()]

if missing:
    raise RuntimeError(
        "Missing required data: " + ", ".join(missing)
    )

NSL_CLASS_ORDER = [
    "Normal",
    "DoS",
    "Probe",
    "R2L",
    "U2R"
]

NSL_ATTACK_FAMILIES = {
    "Normal": [
        "normal"
    ],
    "DoS": [
        "apache2", "back", "land", "mailbomb",
        "neptune", "pod", "processtable", "smurf",
        "teardrop", "udpstorm", "worm"
    ],
    "Probe": [
        "ipsweep", "mscan", "nmap",
        "portsweep", "saint", "satan"
    ],
    "R2L": [
        "ftp_write", "guess_passwd", "httptunnel",
        "imap", "multihop", "named", "phf",
        "sendmail", "snmpgetattack", "snmpguess",
        "spy", "warezclient", "warezmaster",
        "xlock", "xsnoop"
    ],
    "U2R": [
        "buffer_overflow", "loadmodule", "perl",
        "ps", "rootkit", "sqlattack", "xterm"
    ]
}

all_configured = [
    attack
    for attacks in NSL_ATTACK_FAMILIES.values()
    for attack in attacks
]

duplicates = sorted(
    name for name, count in Counter(all_configured).items()
    if count != 1
)

if duplicates:
    raise RuntimeError(
        f"Duplicate attack assignments: {duplicates}"
    )

NSL_ATTACK_TO_FAMILY = {
    attack: family
    for family, attacks in NSL_ATTACK_FAMILIES.items()
    for attack in attacks
}

observed_attacks = (
    set(NSL_TRAIN_ATTACKS)
    | set(NSL_TEST_ATTACKS)
)

configured_attacks = set(NSL_ATTACK_TO_FAMILY)

unmapped = sorted(observed_attacks - configured_attacks)
unused = sorted(configured_attacks - observed_attacks)

print("\nMAPPING COVERAGE")
print("Observed attack labels:", len(observed_attacks))
print("Configured attack labels:", len(configured_attacks))
print("Unmapped attack labels:", unmapped)
print("Unused mapping entries:", unused)

if unmapped or unused:
    raise RuntimeError("Attack mapping coverage is incomplete.")

if len(observed_attacks) != 40:
    raise RuntimeError("Expected 40 original labels.")

NSL_FAMILY_TRAIN = NSL_TRAIN_ATTACKS.map(
    NSL_ATTACK_TO_FAMILY
)

NSL_FAMILY_TEST = NSL_TEST_ATTACKS.map(
    NSL_ATTACK_TO_FAMILY
)

if NSL_FAMILY_TRAIN.isna().any():
    raise RuntimeError("Unmapped training labels.")

if NSL_FAMILY_TEST.isna().any():
    raise RuntimeError("Unmapped testing labels.")

distribution = pd.DataFrame({
    "Training": NSL_FAMILY_TRAIN.value_counts(),
    "Official Test": NSL_FAMILY_TEST.value_counts()
}).reindex(NSL_CLASS_ORDER, fill_value=0)

distribution.index.name = "Class"

print("\nFIVE-CLASS DISTRIBUTION")
print(distribution.to_string())

print("\nCLASS TOTALS")
print("Training:", int(distribution["Training"].sum()))
print("Testing:", int(distribution["Official Test"].sum()))

if int(distribution["Training"].sum()) != 125973:
    raise RuntimeError("Training class total mismatch.")

if int(distribution["Official Test"].sum()) != 22544:
    raise RuntimeError("Testing class total mismatch.")

if not (distribution > 0).all().all():
    raise RuntimeError(
        "One or more classes have zero observations."
    )

if (
    distribution["Training"].to_dict()
    != {
        "Normal": 67343,
        "DoS": 45927,
        "Probe": 11656,
        "R2L": 995,
        "U2R": 52
    }
):
    raise RuntimeError(
        "Training family counts differ from NSL-KDD reference."
    )

NSL_CLASS_TO_INDEX = {
    name: index
    for index, name in enumerate(NSL_CLASS_ORDER)
}

NSL_MAPPING_VERIFIED = True

print("\nCLASS ENCODING")
for name, index in NSL_CLASS_TO_INDEX.items():
    print(f"{index}: {name}")

print("\nCELL 04: PASSED")
print("All observed attacks mapped: YES")
print("Original records modified: NO")
print("=" * 72)


CELL 04 — FIVE-CLASS ATTACK MAPPING

MAPPING COVERAGE
Observed attack labels: 40
Configured attack labels: 40
Unmapped attack labels: []
Unused mapping entries: []

FIVE-CLASS DISTRIBUTION
        Training  Official Test
Class                          
Normal     67343           9711
DoS        45927           7460
Probe      11656           2421
R2L          995           2885
U2R           52             67

CLASS TOTALS
Training: 125973
Testing: 22544

CLASS ENCODING
0: Normal
1: DoS
2: Probe
3: R2L
4: U2R

CELL 04: PASSED
All observed attacks mapped: YES
Original records modified: NO


In [5]:

# ============================================================
# NOTEBOOK 09 — NSL-KDD PREPROCESSING
# CELL 05 — STRATIFIED DEVELOPMENT SPLIT
# ============================================================

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

if not globals().get("NSL_MAPPING_VERIFIED", False):
    raise RuntimeError("Cell 04 must pass first.")

print("=" * 72)
print("CELL 05 — STRATIFIED DEVELOPMENT SPLIT")
print("=" * 72)

NSL_RANDOM_SEED = 42
NSL_VALIDATION_RATIO = 0.15

all_indices = np.arange(
    len(NSL_RAW_TRAIN),
    dtype=np.int64
)

train_indices, validation_indices = train_test_split(
    all_indices,
    test_size=NSL_VALIDATION_RATIO,
    random_state=NSL_RANDOM_SEED,
    stratify=NSL_FAMILY_TRAIN.to_numpy(),
    shuffle=True
)

NSL_TRAIN_INDICES = np.sort(train_indices)
NSL_VALIDATION_INDICES = np.sort(validation_indices)

if len(np.intersect1d(
    NSL_TRAIN_INDICES,
    NSL_VALIDATION_INDICES
)) != 0:
    raise RuntimeError("Development partitions overlap.")

combined = np.sort(
    np.concatenate([
        NSL_TRAIN_INDICES,
        NSL_VALIDATION_INDICES
    ])
)

if not np.array_equal(combined, all_indices):
    raise RuntimeError(
        "Development partitions do not cover official training data."
    )

NSL_DEV_TRAIN = (
    NSL_RAW_TRAIN.iloc[NSL_TRAIN_INDICES]
    .reset_index(drop=True)
    .copy()
)

NSL_DEV_VALIDATION = (
    NSL_RAW_TRAIN.iloc[NSL_VALIDATION_INDICES]
    .reset_index(drop=True)
    .copy()
)

NSL_DEV_TRAIN_FAMILY = (
    NSL_FAMILY_TRAIN.iloc[NSL_TRAIN_INDICES]
    .reset_index(drop=True)
)

NSL_DEV_VALIDATION_FAMILY = (
    NSL_FAMILY_TRAIN.iloc[NSL_VALIDATION_INDICES]
    .reset_index(drop=True)
)

NSL_OFFICIAL_TEST_FAMILY = (
    NSL_FAMILY_TEST.reset_index(drop=True)
)

if len(NSL_DEV_TRAIN) != 107077:
    raise RuntimeError("Unexpected development training size.")

if len(NSL_DEV_VALIDATION) != 18896:
    raise RuntimeError("Unexpected validation size.")

if len(NSL_RAW_TEST) != 22544:
    raise RuntimeError("Unexpected official test size.")

class_distribution = pd.DataFrame({
    "Development Train":
        NSL_DEV_TRAIN_FAMILY.value_counts(),
    "Validation":
        NSL_DEV_VALIDATION_FAMILY.value_counts(),
    "Official Test":
        NSL_OFFICIAL_TEST_FAMILY.value_counts()
}).reindex(NSL_CLASS_ORDER, fill_value=0)

print("\nPARTITION SIZES")
print("Development training:", len(NSL_DEV_TRAIN))
print("Development validation:", len(NSL_DEV_VALIDATION))
print("Official test:", len(NSL_RAW_TEST))

print("\nCLASS DISTRIBUTION")
print(class_distribution.to_string())

print("\nPARTITION VERIFICATION")
print(
    "Training and validation overlap:",
    len(np.intersect1d(
        NSL_TRAIN_INDICES,
        NSL_VALIDATION_INDICES
    ))
)
print(
    "Training + validation:",
    len(NSL_DEV_TRAIN) + len(NSL_DEV_VALIDATION)
)
print("Original official training:", len(NSL_RAW_TRAIN))

for class_name in NSL_CLASS_ORDER:
    if (
        class_distribution.loc[class_name, "Development Train"] < 1
        or class_distribution.loc[class_name, "Validation"] < 1
        or class_distribution.loc[class_name, "Official Test"] < 1
    ):
        raise RuntimeError(
            f"Insufficient class support: {class_name}"
        )

NSL_SPLIT_VERIFIED = True

print("\nCELL 05: PASSED")
print("Stratified split: VERIFIED")
print("Official test membership unchanged: YES")
print("=" * 72)


CELL 05 — STRATIFIED DEVELOPMENT SPLIT

PARTITION SIZES
Development training: 107077
Development validation: 18896
Official test: 22544

CLASS DISTRIBUTION
        Development Train  Validation  Official Test
attack                                              
Normal              57241       10102           9711
DoS                 39038        6889           7460
Probe                9908        1748           2421
R2L                   846         149           2885
U2R                    44           8             67

PARTITION VERIFICATION
Training and validation overlap: 0
Training + validation: 125973
Original official training: 125973

CELL 05: PASSED
Stratified split: VERIFIED
Official test membership unchanged: YES


In [6]:

# ============================================================
# NOTEBOOK 09 — NSL-KDD PREPROCESSING
# CELL 06 — FEATURE ENCODING CONFIGURATION
# ============================================================

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

if not globals().get("NSL_SPLIT_VERIFIED", False):
    raise RuntimeError("Cell 05 must pass first.")

print("=" * 72)
print("CELL 06 — FEATURE ENCODING CONFIGURATION")
print("=" * 72)

NSL_CATEGORICAL_COLUMNS = [
    "protocol_type",
    "service",
    "flag"
]

NSL_NUMERIC_COLUMNS = [
    column
    for column in NSL_FEATURE_COLUMNS
    if column not in NSL_CATEGORICAL_COLUMNS
]

assert len(NSL_FEATURE_COLUMNS) == 41
assert len(NSL_CATEGORICAL_COLUMNS) == 3
assert len(NSL_NUMERIC_COLUMNS) == 38

if len(set(NSL_FEATURE_COLUMNS)) != 41:
    raise RuntimeError("Duplicate feature names.")

excluded_columns = {
    "attack",
    "difficulty",
    "attack_family",
    "Label"
}

if set(NSL_FEATURE_COLUMNS) & excluded_columns:
    raise RuntimeError(
        "Target or auxiliary columns included as predictors."
    )

for name, data in {
    "Development Train": NSL_DEV_TRAIN,
    "Validation": NSL_DEV_VALIDATION,
    "Official Test": NSL_RAW_TEST
}.items():
    missing_columns = (
        set(NSL_FEATURE_COLUMNS) - set(data.columns)
    )

    if missing_columns:
        raise RuntimeError(
            f"{name}: missing predictors {missing_columns}"
        )

# Encoding pipeline

NSL_FEATURE_PIPELINE = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
                dtype=np.float32
            ),
            NSL_CATEGORICAL_COLUMNS
        ),
        (
            "numeric",
            "passthrough",
            NSL_NUMERIC_COLUMNS
        )
    ],
    remainder="drop",
    sparse_threshold=0.0,
    verbose_feature_names_out=False
)

NSL_PREPROCESSING_CONFIG = {
    "dataset": "NSL-KDD",
    "raw_feature_count": 41,
    "categorical_features": NSL_CATEGORICAL_COLUMNS,
    "numeric_features": NSL_NUMERIC_COLUMNS,
    "encoding": "OneHotEncoder",
    "unknown_category_handling": "ignore",
    "numeric_transformation": "passthrough",
    "numeric_scaling": False,
    "excluded_columns": ["attack", "difficulty"],
    "encoder_fit_partition": "Development Train",
    "validation_ratio": NSL_VALIDATION_RATIO,
    "random_seed": NSL_RANDOM_SEED,
    "class_order": NSL_CLASS_ORDER,
    "resampling": "None"
}

print("\nFEATURE CONFIGURATION")
print("Original predictors:", len(NSL_FEATURE_COLUMNS))
print("Categorical predictors:", NSL_CATEGORICAL_COLUMNS)
print("Numerical predictors:", len(NSL_NUMERIC_COLUMNS))
print("Encoder:", "OneHotEncoder")
print("Unknown categories:", "ignore")
print("Numerical scaling:", "Disabled")
print("Resampling:", "None")
print("Encoder fitted:", hasattr(NSL_FEATURE_PIPELINE, "transformers_"))

print("\nTRAINING CATEGORICAL CARDINALITIES")

for column in NSL_CATEGORICAL_COLUMNS:
    cardinality = NSL_DEV_TRAIN[column].nunique(dropna=False)
    print(f"{column:<20}: {cardinality}")

print("\nDATA BOUNDARIES")
print("Encoder fitting data: Development Train")
print("Validation transformation: After fitting")
print("Official test transformation: After fitting")
print("Target included in predictors: NO")
print("Difficulty included in predictors: NO")

NSL_PREPROCESSING_CONFIG_VERIFIED = True

print("\nCELL 06: PASSED")
print("Encoding pipeline defined: YES")
print("Encoding pipeline fitted: NO")
print("Model training performed: NO")
print("=" * 72)


CELL 06 — FEATURE ENCODING CONFIGURATION

FEATURE CONFIGURATION
Original predictors: 41
Categorical predictors: ['protocol_type', 'service', 'flag']
Numerical predictors: 38
Encoder: OneHotEncoder
Unknown categories: ignore
Numerical scaling: Disabled
Resampling: None
Encoder fitted: False

TRAINING CATEGORICAL CARDINALITIES
protocol_type       : 3
service             : 70
flag                : 11

DATA BOUNDARIES
Encoder fitting data: Development Train
Validation transformation: After fitting
Official test transformation: After fitting
Target included in predictors: NO
Difficulty included in predictors: NO

CELL 06: PASSED
Encoding pipeline defined: YES
Encoding pipeline fitted: NO
Model training performed: NO


In [7]:

# ============================================================
# NOTEBOOK 09 — NSL-KDD PREPROCESSING
# CELL 07 — TRAINING-DERIVED FEATURE ENCODER
# ============================================================

import numpy as np
from sklearn.utils.validation import check_is_fitted

print("=" * 72)
print("CELL 07 — TRAINING-DERIVED FEATURE ENCODER")
print("=" * 72)

if not globals().get("NSL_PREPROCESSING_CONFIG_VERIFIED", False):
    raise RuntimeError("Cell 06 must pass first.")

NSL_ENCODER_FITTED = False

X_fit = NSL_DEV_TRAIN.loc[:, NSL_FEATURE_COLUMNS]

if X_fit.shape != (107077, 41):
    raise RuntimeError(
        f"Unexpected encoder fitting data shape: {X_fit.shape}"
    )

NSL_FEATURE_PIPELINE.fit(X_fit)

check_is_fitted(NSL_FEATURE_PIPELINE)

categorical_encoder = (
    NSL_FEATURE_PIPELINE.named_transformers_["categorical"]
)

NSL_ENCODER_CATEGORIES = {
    column: [
        str(value) for value in categories
    ]
    for column, categories in zip(
        NSL_CATEGORICAL_COLUMNS,
        categorical_encoder.categories_
    )
}

NSL_ENCODED_FEATURE_NAMES = list(
    NSL_FEATURE_PIPELINE.get_feature_names_out()
)

categorical_feature_count = sum(
    len(categories)
    for categories in categorical_encoder.categories_
)

numeric_feature_count = len(NSL_NUMERIC_COLUMNS)

NSL_ENCODED_FEATURE_COUNT = len(
    NSL_ENCODED_FEATURE_NAMES
)

if categorical_feature_count != 84:
    raise RuntimeError(
        f"Unexpected categorical feature count: "
        f"{categorical_feature_count}"
    )

if NSL_ENCODED_FEATURE_COUNT != 122:
    raise RuntimeError(
        f"Unexpected encoded feature count: "
        f"{NSL_ENCODED_FEATURE_COUNT}"
    )

if len(set(NSL_ENCODED_FEATURE_NAMES)) != 122:
    raise RuntimeError("Duplicate encoded feature names.")

if any(
    name in NSL_ENCODED_FEATURE_NAMES
    for name in ["attack", "difficulty", "Label"]
):
    raise RuntimeError(
        "Target or difficulty column found in features."
    )

if categorical_encoder.handle_unknown != "ignore":
    raise RuntimeError(
        "Unexpected unknown-category handling."
    )

print("\nENCODER CONFIGURATION")
print("Fitting partition: Development Train")
print("Fitting observations:", len(X_fit))
print("Original predictors:", len(NSL_FEATURE_COLUMNS))
print("Categorical predictors:", len(NSL_CATEGORICAL_COLUMNS))
print("Numerical predictors:", numeric_feature_count)
print("Unknown categories:", categorical_encoder.handle_unknown)

print("\nENCODED FEATURE COUNTS")

for column, categories in NSL_ENCODER_CATEGORIES.items():
    print(f"{column:<20}: {len(categories)}")

print("Total categorical features:", categorical_feature_count)
print("Numerical features:", numeric_feature_count)
print("Total encoded features:", NSL_ENCODED_FEATURE_COUNT)

print("\nFIRST 12 ENCODED FEATURE NAMES")

for name in NSL_ENCODED_FEATURE_NAMES[:12]:
    print(name)

NSL_ENCODER_FITTED = True

print("\nVERIFICATION")
print("Encoder fitted: PASS")
print("Feature count: PASS")
print("Feature-name uniqueness: PASS")
print("Unknown-category configuration: PASS")
print("CELL 07: PASSED")
print("Random Forest training performed: NO")
print("=" * 72)


CELL 07 — TRAINING-DERIVED FEATURE ENCODER

ENCODER CONFIGURATION
Fitting partition: Development Train
Fitting observations: 107077
Original predictors: 41
Categorical predictors: 3
Numerical predictors: 38
Unknown categories: ignore

ENCODED FEATURE COUNTS
protocol_type       : 3
service             : 70
flag                : 11
Total categorical features: 84
Numerical features: 38
Total encoded features: 122

FIRST 12 ENCODED FEATURE NAMES
protocol_type_icmp
protocol_type_tcp
protocol_type_udp
service_IRC
service_X11
service_Z39_50
service_aol
service_auth
service_bgp
service_courier
service_csnet_ns
service_ctf

VERIFICATION
Encoder fitted: PASS
Feature count: PASS
Feature-name uniqueness: PASS
Unknown-category configuration: PASS
CELL 07: PASSED
Random Forest training performed: NO


In [8]:

# ============================================================
# NOTEBOOK 09 — NSL-KDD PREPROCESSING
# CELL 08 — DATASET TRANSFORMATION AND TARGET ENCODING
# ============================================================

import numpy as np
import pandas as pd

print("=" * 72)
print("CELL 08 — DATASET TRANSFORMATION")
print("=" * 72)

if not globals().get("NSL_ENCODER_FITTED", False):
    raise RuntimeError("Cell 07 must pass first.")

NSL_TRANSFORM_VERIFIED = False

def transform_partition(dataframe, family_labels, name):
    features = dataframe.loc[:, NSL_FEATURE_COLUMNS]

    transformed = NSL_FEATURE_PIPELINE.transform(features)

    X = np.asarray(transformed, dtype=np.float32)

    y_series = family_labels.map(NSL_CLASS_TO_INDEX)

    if y_series.isna().any():
        raise RuntimeError(
            f"{name}: unmapped target classes."
        )

    y = y_series.to_numpy(dtype=np.int8)

    if X.shape != (len(dataframe), NSL_ENCODED_FEATURE_COUNT):
        raise RuntimeError(
            f"{name}: invalid transformed shape {X.shape}"
        )

    if y.shape != (len(dataframe),):
        raise RuntimeError(
            f"{name}: invalid target shape {y.shape}"
        )

    if not np.isfinite(X).all():
        raise RuntimeError(
            f"{name}: non-finite feature values."
        )

    if np.any(y < 0) or np.any(y >= len(NSL_CLASS_ORDER)):
        raise RuntimeError(
            f"{name}: invalid encoded class indices."
        )

    if len(np.unique(y)) != len(NSL_CLASS_ORDER):
        raise RuntimeError(
            f"{name}: missing target class."
        )

    return X, y

NSL_X_TRAIN, NSL_Y_TRAIN = transform_partition(
    NSL_DEV_TRAIN,
    NSL_DEV_TRAIN_FAMILY,
    "Development Train"
)

NSL_X_VALIDATION, NSL_Y_VALIDATION = transform_partition(
    NSL_DEV_VALIDATION,
    NSL_DEV_VALIDATION_FAMILY,
    "Validation"
)

NSL_X_TEST, NSL_Y_TEST = transform_partition(
    NSL_RAW_TEST,
    NSL_OFFICIAL_TEST_FAMILY,
    "Official Test"
)

NSL_TRANSFORMED_DATA = {
    "Development Train": (NSL_X_TRAIN, NSL_Y_TRAIN),
    "Validation": (NSL_X_VALIDATION, NSL_Y_VALIDATION),
    "Official Test": (NSL_X_TEST, NSL_Y_TEST)
}

print("\nTRANSFORMED DATASET SUMMARY")

for name, (X, y) in NSL_TRANSFORMED_DATA.items():

    print("\n" + "-" * 72)
    print(name)
    print("-" * 72)

    print("Feature matrix:", X.shape)
    print("Target vector:", y.shape)
    print("Feature dtype:", X.dtype)
    print("Target dtype:", y.dtype)
    print("Feature values finite:", bool(np.isfinite(X).all()))
    print("Target classes:", len(np.unique(y)))

expected_rows = {
    "Development Train": 107077,
    "Validation": 18896,
    "Official Test": 22544
}

for name, (X, y) in NSL_TRANSFORMED_DATA.items():

    if X.shape != (
        expected_rows[name],
        NSL_ENCODED_FEATURE_COUNT
    ):
        raise RuntimeError(
            f"{name}: unexpected feature matrix shape."
        )

    if X.dtype != np.float32:
        raise RuntimeError(
            f"{name}: feature dtype must be float32."
        )

    if len(y) != expected_rows[name]:
        raise RuntimeError(
            f"{name}: unexpected target count."
        )

if not (
    NSL_X_TRAIN.shape[1]
    == NSL_X_VALIDATION.shape[1]
    == NSL_X_TEST.shape[1]
):
    raise RuntimeError(
        "Inconsistent encoded feature dimensions."
    )

NSL_TRANSFORM_VERIFIED = True

print("\nVERIFICATION")
print("Consistent feature schema: PASS")
print("Finite feature values: PASS")
print("Target mapping: PASS")
print("Original partition sizes: PASS")
print("CELL 08: PASSED")
print("Random Forest training performed: NO")
print("=" * 72)


CELL 08 — DATASET TRANSFORMATION

TRANSFORMED DATASET SUMMARY

------------------------------------------------------------------------
Development Train
------------------------------------------------------------------------
Feature matrix: (107077, 122)
Target vector: (107077,)
Feature dtype: float32
Target dtype: int8
Feature values finite: True
Target classes: 5

------------------------------------------------------------------------
Validation
------------------------------------------------------------------------
Feature matrix: (18896, 122)
Target vector: (18896,)
Feature dtype: float32
Target dtype: int8
Feature values finite: True
Target classes: 5

------------------------------------------------------------------------
Official Test
------------------------------------------------------------------------
Feature matrix: (22544, 122)
Target vector: (22544,)
Feature dtype: float32
Target dtype: int8
Feature values finite: True
Target classes: 5

VERIFICATION
Consistent feat

In [9]:

# ============================================================
# NOTEBOOK 09 — NSL-KDD PREPROCESSING
# CELL 09 — TRANSFORMATION INTEGRITY VERIFICATION
# ============================================================

import numpy as np
import pandas as pd

print("=" * 72)
print("CELL 09 — TRANSFORMATION INTEGRITY VERIFICATION")
print("=" * 72)

if not globals().get("NSL_TRANSFORM_VERIFIED", False):
    raise RuntimeError("Cell 08 must pass first.")

NSL_INTEGRITY_VERIFIED = False

partitions = {
    "Development Train": (
        NSL_DEV_TRAIN,
        NSL_DEV_TRAIN_FAMILY,
        NSL_X_TRAIN,
        NSL_Y_TRAIN
    ),
    "Validation": (
        NSL_DEV_VALIDATION,
        NSL_DEV_VALIDATION_FAMILY,
        NSL_X_VALIDATION,
        NSL_Y_VALIDATION
    ),
    "Official Test": (
        NSL_RAW_TEST,
        NSL_OFFICIAL_TEST_FAMILY,
        NSL_X_TEST,
        NSL_Y_TEST
    )
}

encoder = NSL_FEATURE_PIPELINE.named_transformers_["categorical"]

categorical_width = sum(
    len(categories)
    for categories in encoder.categories_
)

numeric_width = len(NSL_NUMERIC_COLUMNS)

if categorical_width + numeric_width != 122:
    raise RuntimeError(
        "Unexpected feature-block dimensions."
    )

print("\nCATEGORICAL ENCODING AUDIT")

unknown_records = []

for partition_name, (raw, family, X, y) in partitions.items():

    offset = 0

    for column, categories in zip(
        NSL_CATEGORICAL_COLUMNS,
        encoder.categories_
    ):

        width = len(categories)

        block = X[:, offset:offset + width]

        if block.shape != (len(raw), width):
            raise RuntimeError(
                f"{partition_name}: invalid block for {column}."
            )

        if not np.all(
            (block == 0.0) | (block == 1.0)
        ):
            raise RuntimeError(
                f"{partition_name}: invalid one-hot values."
            )

        raw_values = raw[column].astype(str).to_numpy()
        known_categories = np.asarray(
            categories,
            dtype=str
        )

        is_known = np.isin(
            raw_values,
            known_categories
        )

        observed_sums = block.sum(axis=1)
        expected_sums = is_known.astype(np.float32)

        if not np.array_equal(
            observed_sums,
            expected_sums
        ):
            raise RuntimeError(
                f"{partition_name}: one-hot encoding mismatch "
                f"for {column}."
            )

        unknown_count = int(
            np.count_nonzero(~is_known)
        )

        unknown_records.append({
            "Partition": partition_name,
            "Feature": column,
            "Known Categories": len(categories),
            "Unknown Records": unknown_count
        })

        offset += width

    # Verify original numerical features after float32 conversion

    numeric_block = X[
        :,
        categorical_width:
        categorical_width + numeric_width
    ]

    raw_numeric = raw.loc[
        :, NSL_NUMERIC_COLUMNS
    ].to_numpy(dtype=np.float32)

    if not np.array_equal(
        numeric_block,
        raw_numeric
    ):
        raise RuntimeError(
            f"{partition_name}: numeric feature mismatch."
        )

    expected_targets = (
        family.map(NSL_CLASS_TO_INDEX)
        .to_numpy(dtype=np.int8)
    )

    if not np.array_equal(y, expected_targets):
        raise RuntimeError(
            f"{partition_name}: target mapping mismatch."
        )

NSL_UNKNOWN_CATEGORY_AUDIT = pd.DataFrame(
    unknown_records
)

print(
    NSL_UNKNOWN_CATEGORY_AUDIT.to_string(
        index=False
    )
)

print("\nCLASS DISTRIBUTION VERIFICATION")

class_distributions = {}

for partition_name, (_, _, X, y) in partitions.items():

    counts = np.bincount(
        y.astype(np.int64),
        minlength=5
    )

    class_distributions[partition_name] = counts

NSL_PROCESSED_CLASS_DISTRIBUTION = pd.DataFrame(
    class_distributions,
    index=NSL_CLASS_ORDER
)

NSL_PROCESSED_CLASS_DISTRIBUTION.index.name = "Class"

print(
    NSL_PROCESSED_CLASS_DISTRIBUTION.to_string()
)

expected_train = np.array(
    [57241, 39038, 9908, 846, 44]
)

expected_validation = np.array(
    [10102, 6889, 1748, 149, 8]
)

expected_test = np.array(
    [9711, 7460, 2421, 2885, 67]
)

if not np.array_equal(
    class_distributions["Development Train"],
    expected_train
):
    raise RuntimeError(
        "Development training class distribution mismatch."
    )

if not np.array_equal(
    class_distributions["Validation"],
    expected_validation
):
    raise RuntimeError(
        "Validation class distribution mismatch."
    )

if not np.array_equal(
    class_distributions["Official Test"],
    expected_test
):
    raise RuntimeError(
        "Official test class distribution mismatch."
    )

if len(np.intersect1d(
    NSL_TRAIN_INDICES,
    NSL_VALIDATION_INDICES
)) != 0:
    raise RuntimeError(
        "Development training and validation indices overlap."
    )

if (
    len(NSL_TRAIN_INDICES)
    + len(NSL_VALIDATION_INDICES)
    != 125973
):
    raise RuntimeError(
        "Official training partition coverage mismatch."
    )

if NSL_X_TEST.shape[0] != 22544:
    raise RuntimeError(
        "Official test partition size mismatch."
    )

NSL_INTEGRITY_VERIFIED = True

print("\nINTEGRITY SUMMARY")
print("Encoded feature count:", NSL_ENCODED_FEATURE_COUNT)
print("One-hot encoding rules: PASS")
print("Unknown-category handling: PASS")
print("Numerical feature preservation: PASS")
print("Target class mapping: PASS")
print("Training-validation separation: PASS")
print("Official test partition: PASS")
print("CELL 09: PASSED")
print("Random Forest training performed: NO")
print("=" * 72)


CELL 09 — TRANSFORMATION INTEGRITY VERIFICATION

CATEGORICAL ENCODING AUDIT
        Partition       Feature  Known Categories  Unknown Records
Development Train protocol_type                 3                0
Development Train       service                70                0
Development Train          flag                11                0
       Validation protocol_type                 3                0
       Validation       service                70                0
       Validation          flag                11                0
    Official Test protocol_type                 3                0
    Official Test       service                70                0
    Official Test          flag                11                0

CLASS DISTRIBUTION VERIFICATION
        Development Train  Validation  Official Test
Class                                               
Normal              57241       10102           9711
DoS                 39038        6889           7460
Probe    

In [11]:

# ============================================================
# NOTEBOOK 09 — NSL-KDD PREPROCESSING
# CELL 10 — PROCESSED DATA AND ARTEFACT EXPORT
# ============================================================

from pathlib import Path
import io
import json
import hashlib
import joblib
import numpy as np
import pandas as pd

print("=" * 72)
print("CELL 10 — PROCESSED DATA AND ARTEFACT EXPORT")
print("=" * 72)

if not globals().get("NSL_INTEGRITY_VERIFIED", False):
    raise RuntimeError("Cell 09 must pass first.")

PROJECT_ROOT = Path("/content/drive/MyDrive/MSc_Project")

NSL_PROCESSED_DIR = (
    PROJECT_ROOT / "data/NSL_KDD/processed"
)

NSL_RESULTS_DIR_09 = (
    PROJECT_ROOT / "results/NSL_KDD_PREPROCESSING"
)

NSL_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
NSL_RESULTS_DIR_09.mkdir(parents=True, exist_ok=True)

NSL09_EXPORT_COMPLETE = False
NSL09_SAVED_FILES = []

def ns09_write_once(path, content):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    if not isinstance(content, bytes):
        raise TypeError("Expected bytes.")

    try:
        with path.open("xb") as handle:
            handle.write(content)

        status = "CREATED"

    except FileExistsError:
        existing_hash = hashlib.sha256(
            path.read_bytes()
        ).digest()

        new_hash = hashlib.sha256(content).digest()

        if existing_hash != new_hash:
            raise RuntimeError(
                f"Existing artefact differs: {path}"
            )

        status = "ALREADY VERIFIED"

    return status

def ns09_save(path, content):
    status = ns09_write_once(path, content)
    NSL09_SAVED_FILES.append(Path(path))
    print(f"{Path(path).name}: {status}")

def ns09_parquet_bytes(features, targets):
    frame = pd.DataFrame(
        features,
        columns=NSL_ENCODED_FEATURE_NAMES,
        copy=False
    )

    frame["Label"] = np.asarray(
        targets,
        dtype=np.int8
    )

    buffer = io.BytesIO()

    frame.to_parquet(
        buffer,
        index=False,
        engine="pyarrow",
        compression="zstd"
    )

    return buffer.getvalue()

# Export processed partitions

partitions = {
    "nslkdd_train.parquet": (
        NSL_X_TRAIN, NSL_Y_TRAIN
    ),
    "nslkdd_validation.parquet": (
        NSL_X_VALIDATION, NSL_Y_VALIDATION
    ),
    "nslkdd_test.parquet": (
        NSL_X_TEST, NSL_Y_TEST
    )
}

print("\nPROCESSED DATASET EXPORTS")

for filename, (X, y) in partitions.items():
    ns09_save(
        NSL_PROCESSED_DIR / filename,
        ns09_parquet_bytes(X, y)
    )

# Serialize the fitted training-derived encoder

encoder_buffer = io.BytesIO()

joblib.dump(
    NSL_FEATURE_PIPELINE,
    encoder_buffer,
    compress=3
)

print("\nPREPROCESSING ARTEFACTS")

ns09_save(
    NSL_PROCESSED_DIR / "nslkdd_preprocessor.joblib",
    encoder_buffer.getvalue()
)

# Save original training-row partition indices

split_buffer = io.BytesIO()

np.savez_compressed(
    split_buffer,
    train_indices=np.asarray(
        NSL_TRAIN_INDICES, dtype=np.int64
    ),
    validation_indices=np.asarray(
        NSL_VALIDATION_INDICES, dtype=np.int64
    ),
    original_training_size=np.int64(125973),
    random_seed=np.int64(NSL_RANDOM_SEED)
)

ns09_save(
    NSL_PROCESSED_DIR / "nslkdd_split_indices.npz",
    split_buffer.getvalue()
)

# Save the exact encoded feature schema

feature_schema = {
    "dataset": "NSL-KDD",
    "feature_count": len(NSL_ENCODED_FEATURE_NAMES),
    "feature_names": list(NSL_ENCODED_FEATURE_NAMES),
    "feature_dtype": "float32",
    "label_column": "Label",
    "label_dtype": "int8",
    "label_encoding": "integer_class_indices",
    "raw_predictor_count": 41,
    "categorical_input_features": list(
        NSL_CATEGORICAL_COLUMNS
    ),
    "numeric_input_features": list(
        NSL_NUMERIC_COLUMNS
    ),
    "categorical_encoded_feature_count": 84,
    "numeric_feature_count": 38,
    "encoder_categories": NSL_ENCODER_CATEGORIES,
    "encoder_fit_partition": "development_train",
    "unknown_category_handling": "ignore",
    "numeric_scaling": False,
    "resampling": None,
    "excluded_raw_columns": [
        "attack", "difficulty"
    ],
    "partition_records": {
        "train": 107077,
        "validation": 18896,
        "test": 22544
    }
}

schema_content = (
    json.dumps(
        feature_schema,
        indent=2,
        sort_keys=True
    ) + "\n"
).encode("utf-8")

ns09_save(
    NSL_PROCESSED_DIR / "nslkdd_feature_schema.json",
    schema_content
)

# Save five-class target and attack-family mappings

class_mapping = {
    "dataset": "NSL-KDD",
    "class_order": list(NSL_CLASS_ORDER),
    "class_to_index": dict(NSL_CLASS_TO_INDEX),
    "index_to_class": {
        str(index): name
        for index, name in enumerate(NSL_CLASS_ORDER)
    },
    "attack_to_family": dict(
        sorted(NSL_ATTACK_TO_FAMILY.items())
    ),
    "attack_count": len(NSL_ATTACK_TO_FAMILY),
    "target_column": "Label"
}

mapping_content = (
    json.dumps(
        class_mapping,
        indent=2,
        sort_keys=True
    ) + "\n"
).encode("utf-8")

ns09_save(
    NSL_PROCESSED_DIR / "nslkdd_class_mapping.json",
    mapping_content
)

# Export preprocessing audit tables

train_attack_counts = NSL_TRAIN_ATTACKS.value_counts()
test_attack_counts = NSL_TEST_ATTACKS.value_counts()

attack_names = sorted(
    set(train_attack_counts.index)
    | set(test_attack_counts.index)
)

attack_inventory = pd.DataFrame({
    "Attack": attack_names,
    "Family": [
        NSL_ATTACK_TO_FAMILY[name]
        for name in attack_names
    ],
    "Training Count": [
        int(train_attack_counts.get(name, 0))
        for name in attack_names
    ],
    "Testing Count": [
        int(test_attack_counts.get(name, 0))
        for name in attack_names
    ]
})

attack_inventory["Test-Only"] = (
    (attack_inventory["Training Count"] == 0)
    & (attack_inventory["Testing Count"] > 0)
)

print("\nAUDIT TABLE EXPORTS")

audit_tables = {
    "NSL09_attack_inventory.csv":
        attack_inventory,
    "NSL09_class_distributions.csv":
        NSL_PROCESSED_CLASS_DISTRIBUTION.reset_index(),
    "NSL09_unknown_categories.csv":
        NSL_UNKNOWN_CATEGORY_AUDIT
}

for filename, table in audit_tables.items():
    ns09_save(
        NSL_RESULTS_DIR_09 / filename,
        table.to_csv(
            index=False
        ).encode("utf-8")
    )

# Confirm output files


# Verify exported artefacts

EXPECTED_ARTEFACT_COUNT = 10

if len(NSL09_SAVED_FILES) != EXPECTED_ARTEFACT_COUNT:
    raise RuntimeError(
        f"Expected {EXPECTED_ARTEFACT_COUNT} artefacts; "
        f"found {len(NSL09_SAVED_FILES)}."
    )

if len(set(NSL09_SAVED_FILES)) != EXPECTED_ARTEFACT_COUNT:
    raise RuntimeError("Duplicate artefact paths detected.")

for path in NSL09_SAVED_FILES:
    if not path.is_file() or path.stat().st_size == 0:
        raise RuntimeError(
            f"Missing or empty artefact: {path}"
        )

NSL09_EXPORT_COMPLETE = True

print("\nEXPORT SUMMARY")
print("Processed datasets:", len(partitions))
print("Supporting artefacts:", 4)
print("Audit tables:", len(audit_tables))
print("Total files:", len(NSL09_SAVED_FILES))
print("Processed directory:", NSL_PROCESSED_DIR)
print("Audit directory:", NSL_RESULTS_DIR_09)

print("\nCELL 10: PASSED")
print("Existing artefacts overwritten: NO")
print("Random Forest training performed: NO")
print("=" * 72)



CELL 10 — PROCESSED DATA AND ARTEFACT EXPORT

PROCESSED DATASET EXPORTS
nslkdd_train.parquet: ALREADY VERIFIED
nslkdd_validation.parquet: ALREADY VERIFIED
nslkdd_test.parquet: ALREADY VERIFIED

PREPROCESSING ARTEFACTS
nslkdd_preprocessor.joblib: ALREADY VERIFIED
nslkdd_split_indices.npz: ALREADY VERIFIED
nslkdd_feature_schema.json: ALREADY VERIFIED
nslkdd_class_mapping.json: ALREADY VERIFIED

AUDIT TABLE EXPORTS
NSL09_attack_inventory.csv: ALREADY VERIFIED
NSL09_class_distributions.csv: ALREADY VERIFIED
NSL09_unknown_categories.csv: ALREADY VERIFIED

EXPORT SUMMARY
Processed datasets: 3
Supporting artefacts: 4
Audit tables: 3
Total files: 10
Processed directory: /content/drive/MyDrive/MSc_Project/data/NSL_KDD/processed
Audit directory: /content/drive/MyDrive/MSc_Project/results/NSL_KDD_PREPROCESSING

CELL 10: PASSED
Existing artefacts overwritten: NO
Random Forest training performed: NO


In [12]:

# ============================================================
# NOTEBOOK 09 — NSL-KDD PREPROCESSING
# CELL 11 — SAVED ARTEFACT INTEGRITY VERIFICATION
# ============================================================

import json
import joblib
import numpy as np
import pandas as pd

print("=" * 72)
print("CELL 11 — SAVED ARTEFACT INTEGRITY VERIFICATION")
print("=" * 72)

if not globals().get("NSL09_EXPORT_COMPLETE", False):
    raise RuntimeError("Cell 10 must pass first.")

NSL09_RELOAD_VERIFIED = False

schema = json.loads(
    (
        NSL_PROCESSED_DIR / "nslkdd_feature_schema.json"
    ).read_text(encoding="utf-8")
)

mapping = json.loads(
    (
        NSL_PROCESSED_DIR / "nslkdd_class_mapping.json"
    ).read_text(encoding="utf-8")
)

if schema["feature_names"] != list(
    NSL_ENCODED_FEATURE_NAMES
):
    raise RuntimeError(
        "Saved feature schema differs from fitted encoder."
    )

if schema["feature_count"] != 122:
    raise RuntimeError(
        "Saved feature count is incorrect."
    )

if mapping["class_order"] != NSL_CLASS_ORDER:
    raise RuntimeError(
        "Saved class mapping is incorrect."
    )

if mapping["class_to_index"] != NSL_CLASS_TO_INDEX:
    raise RuntimeError(
        "Class index mapping mismatch."
    )

if mapping["attack_to_family"] != NSL_ATTACK_TO_FAMILY:
    raise RuntimeError(
        "Attack-family mapping mismatch."
    )

print("\nSCHEMA AND CLASS MAPPING")
print("Saved feature count:", schema["feature_count"])
print("Saved target column:", schema["label_column"])
print("Saved class order:", mapping["class_order"])
print("Feature schema: PASS")
print("Class mapping: PASS")

# Reload processed datasets

partitions = {
    "Development Train": {
        "file": "nslkdd_train.parquet",
        "X": NSL_X_TRAIN,
        "y": NSL_Y_TRAIN,
        "raw": NSL_DEV_TRAIN
    },
    "Validation": {
        "file": "nslkdd_validation.parquet",
        "X": NSL_X_VALIDATION,
        "y": NSL_Y_VALIDATION,
        "raw": NSL_DEV_VALIDATION
    },
    "Official Test": {
        "file": "nslkdd_test.parquet",
        "X": NSL_X_TEST,
        "y": NSL_Y_TEST,
        "raw": NSL_RAW_TEST
    }
}

NSL09_RELOADED_FRAMES = {}

print("\nPROCESSED DATASET VERIFICATION")

for name, entry in partitions.items():

    path = NSL_PROCESSED_DIR / entry["file"]

    frame = pd.read_parquet(
        path,
        engine="pyarrow"
    )

    expected_X = entry["X"]
    expected_y = entry["y"]

    expected_columns = (
        NSL_ENCODED_FEATURE_NAMES + ["Label"]
    )

    if list(frame.columns) != expected_columns:
        raise RuntimeError(
            f"{name}: saved feature columns differ."
        )

    if frame.shape != (
        len(expected_y), 123
    ):
        raise RuntimeError(
            f"{name}: unexpected saved shape."
        )

    if not all(
        dtype == np.dtype("float32")
        for dtype in frame[
            NSL_ENCODED_FEATURE_NAMES
        ].dtypes
    ):
        raise RuntimeError(
            f"{name}: feature dtype mismatch."
        )

    if frame["Label"].dtype != np.dtype("int8"):
        raise RuntimeError(
            f"{name}: label dtype mismatch."
        )

    saved_X = frame[
        NSL_ENCODED_FEATURE_NAMES
    ].to_numpy(dtype=np.float32)

    saved_y = frame["Label"].to_numpy(dtype=np.int8)

    if not np.array_equal(saved_X, expected_X):
        raise RuntimeError(
            f"{name}: saved feature values differ."
        )

    if not np.array_equal(saved_y, expected_y):
        raise RuntimeError(
            f"{name}: saved class labels differ."
        )

    NSL09_RELOADED_FRAMES[name] = frame

    print(
        f"{name:<20} "
        f"{str(frame.shape):<16} "
        "PASS"
    )

# Reload serialized preprocessing pipeline

reloaded_pipeline = joblib.load(
    NSL_PROCESSED_DIR / "nslkdd_preprocessor.joblib"
)

saved_feature_names = list(
    reloaded_pipeline.get_feature_names_out()
)

if saved_feature_names != NSL_ENCODED_FEATURE_NAMES:
    raise RuntimeError(
        "Reloaded encoder feature-name mismatch."
    )

saved_encoder = (
    reloaded_pipeline.named_transformers_["categorical"]
)

saved_categories = {
    column: [str(value) for value in categories]
    for column, categories in zip(
        NSL_CATEGORICAL_COLUMNS,
        saved_encoder.categories_
    )
}

if saved_categories != NSL_ENCODER_CATEGORIES:
    raise RuntimeError(
        "Reloaded encoder categories differ."
    )

# Verify transformed samples using reloaded pipeline

print("\nENCODER REPRODUCIBILITY")

for name, entry in partitions.items():

    raw = entry["raw"]
    original_X = entry["X"]

    n = len(raw)

    sample_positions = np.unique(
        np.concatenate([
            np.arange(min(8, n)),
            np.arange(max(0, n - 8), n),
            np.linspace(
                0, n - 1, num=13,
                dtype=np.int64
            )
        ])
    )

    sample_raw = raw.iloc[
        sample_positions
    ][NSL_FEATURE_COLUMNS]

    reproduced_X = np.asarray(
        reloaded_pipeline.transform(sample_raw),
        dtype=np.float32
    )

    reference_X = original_X[sample_positions]

    if not np.array_equal(
        reproduced_X,
        reference_X
    ):
        raise RuntimeError(
            f"{name}: reloaded encoder output differs."
        )

    print(
        f"{name:<20} "
        f"Sampled rows: {len(sample_positions):>3} "
        "PASS"
    )

# Reload original training-row split

print("\nDEVELOPMENT SPLIT VERIFICATION")

with np.load(
    NSL_PROCESSED_DIR / "nslkdd_split_indices.npz",
    allow_pickle=False
) as saved_split:

    saved_train_indices = saved_split[
        "train_indices"
    ]

    saved_val_indices = saved_split[
        "validation_indices"
    ]

    original_training_size = int(
        saved_split["original_training_size"]
    )

    saved_seed = int(
        saved_split["random_seed"]
    )

if not np.array_equal(
    saved_train_indices,
    NSL_TRAIN_INDICES
):
    raise RuntimeError(
        "Saved training indices differ."
    )

if not np.array_equal(
    saved_val_indices,
    NSL_VALIDATION_INDICES
):
    raise RuntimeError(
        "Saved validation indices differ."
    )

if original_training_size != 125973:
    raise RuntimeError(
        "Original training size mismatch."
    )

if saved_seed != 42:
    raise RuntimeError(
        "Random seed mismatch."
    )

if np.intersect1d(
    saved_train_indices,
    saved_val_indices
).size != 0:
    raise RuntimeError(
        "Saved development partitions overlap."
    )

if not np.array_equal(
    np.sort(
        np.concatenate([
            saved_train_indices,
            saved_val_indices
        ])
    ),
    np.arange(125973)
):
    raise RuntimeError(
        "Saved indices do not cover original training data."
    )

print("Development training indices:", len(saved_train_indices))
print("Validation indices:", len(saved_val_indices))
print("Random seed:", saved_seed)
print("Partition separation: PASS")

# Verify test and development class supports

print("\nSAVED CLASS DISTRIBUTION")

class_counts = {}

for name, frame in NSL09_RELOADED_FRAMES.items():
    class_counts[name] = np.bincount(
        frame["Label"].to_numpy(dtype=np.int64),
        minlength=5
    )

saved_distribution = pd.DataFrame(
    class_counts,
    index=NSL_CLASS_ORDER
)

print(saved_distribution.to_string())

if not saved_distribution.equals(
    NSL_PROCESSED_CLASS_DISTRIBUTION
):
    raise RuntimeError(
        "Saved class distributions differ."
    )

NSL09_RELOAD_VERIFIED = True

print("\nINTEGRITY SUMMARY")
print("Saved datasets: PASS")
print("Encoded feature values: PASS")
print("Encoded class labels: PASS")
print("Serialized encoder: PASS")
print("Encoder reproducibility: PASS")
print("Saved split indices: PASS")
print("Class distributions: PASS")
print("CELL 11: PASSED")
print("Random Forest training performed: NO")
print("=" * 72)


CELL 11 — SAVED ARTEFACT INTEGRITY VERIFICATION

SCHEMA AND CLASS MAPPING
Saved feature count: 122
Saved target column: Label
Saved class order: ['Normal', 'DoS', 'Probe', 'R2L', 'U2R']
Feature schema: PASS
Class mapping: PASS

PROCESSED DATASET VERIFICATION
Development Train    (107077, 123)    PASS
Validation           (18896, 123)     PASS
Official Test        (22544, 123)     PASS

ENCODER REPRODUCIBILITY
Development Train    Sampled rows:  27 PASS
Validation           Sampled rows:  27 PASS
Official Test        Sampled rows:  27 PASS

DEVELOPMENT SPLIT VERIFICATION
Development training indices: 107077
Validation indices: 18896
Random seed: 42
Partition separation: PASS

SAVED CLASS DISTRIBUTION
        Development Train  Validation  Official Test
Normal              57241       10102           9711
DoS                 39038        6889           7460
Probe                9908        1748           2421
R2L                   846         149           2885
U2R                    44 

In [13]:

# ============================================================
# NOTEBOOK 09 — NSL-KDD PREPROCESSING
# CELL 12 — FINAL AUDIT AND COMPLETION RECORD
# ============================================================

from pathlib import Path
import sys
import json
import hashlib
import numpy as np
import pandas as pd
import sklearn
import pyarrow

print("=" * 72)
print("CELL 12 — FINAL AUDIT AND COMPLETION RECORD")
print("=" * 72)

# ------------------------------------------------------------
# 1. Verify previous preprocessing stages
# ------------------------------------------------------------

if not globals().get("NSL09_RELOAD_VERIFIED", False):
    raise RuntimeError("Cell 11 must pass first.")

if not globals().get("NSL09_EXPORT_COMPLETE", False):
    raise RuntimeError("Cell 10 must pass first.")

if len(NSL09_SAVED_FILES) != 10:
    raise RuntimeError(
        "Expected 10 preprocessing artefacts; "
        f"found {len(NSL09_SAVED_FILES)}."
    )

if len(set(NSL09_SAVED_FILES)) != 10:
    raise RuntimeError("Duplicate artefact paths detected.")

# ------------------------------------------------------------
# 2. SHA-256 fingerprint utility
# ------------------------------------------------------------

def ns09_sha256(path):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        for chunk in iter(
            lambda: handle.read(8 * 1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()

# ------------------------------------------------------------
# 3. Verify original NSL-KDD source files
# ------------------------------------------------------------

source_paths = {
    "official_training": Path(NSL_TRAIN_SOURCE),
    "official_test": Path(NSL_TEST_SOURCE)
}

source_records = {}

print("\nORIGINAL SOURCE FINGERPRINTS")

for name, path in source_paths.items():

    if not path.is_file():
        raise FileNotFoundError(path)

    if path.stat().st_size == 0:
        raise RuntimeError(
            f"Empty original source file: {path}"
        )

    source_records[name] = {
        "path": str(path),
        "size_bytes": path.stat().st_size,
        "sha256": ns09_sha256(path)
    }

    print(f"\n{name}")
    print("Path:", path)
    print("SHA-256:", source_records[name]["sha256"])

# ------------------------------------------------------------
# 4. Verify exported preprocessing artefacts
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("PROCESSED ARTEFACT FINGERPRINTS")
print("=" * 72)

artefact_records = {}

for path in sorted(NSL09_SAVED_FILES):

    path = Path(path)

    if not path.is_file():
        raise FileNotFoundError(path)

    if path.stat().st_size == 0:
        raise RuntimeError(
            f"Empty preprocessing artefact: {path}"
        )

    relative_path = str(
        path.relative_to(PROJECT_ROOT)
    )

    artefact_records[relative_path] = {
        "size_bytes": path.stat().st_size,
        "sha256": ns09_sha256(path)
    }

    print(
        f"{path.name:<38} "
        f"{artefact_records[relative_path]['sha256'][:16]}..."
    )

if len(artefact_records) != 10:
    raise RuntimeError(
        "Expected ten unique artefact fingerprints."
    )

# ------------------------------------------------------------
# 5. Verify required preprocessing files
# ------------------------------------------------------------

required_processed = [
    "nslkdd_train.parquet",
    "nslkdd_validation.parquet",
    "nslkdd_test.parquet",
    "nslkdd_preprocessor.joblib",
    "nslkdd_split_indices.npz",
    "nslkdd_feature_schema.json",
    "nslkdd_class_mapping.json"
]

required_audit = [
    "NSL09_attack_inventory.csv",
    "NSL09_class_distributions.csv",
    "NSL09_unknown_categories.csv"
]

expected_paths = {
    *(NSL_PROCESSED_DIR / name for name in required_processed),
    *(NSL_RESULTS_DIR_09 / name for name in required_audit)
}

saved_paths = set(
    Path(path) for path in NSL09_SAVED_FILES
)

if saved_paths != expected_paths:
    raise RuntimeError(
        "Saved artefact paths differ from required files."
    )

# ------------------------------------------------------------
# 6. Verify processed partition metadata
# ------------------------------------------------------------

partition_specifications = {
    "train": {
        "filename": "nslkdd_train.parquet",
        "records": 107077
    },
    "validation": {
        "filename": "nslkdd_validation.parquet",
        "records": 18896
    },
    "test": {
        "filename": "nslkdd_test.parquet",
        "records": 22544
    }
}

print("\n" + "=" * 72)
print("PROCESSED DATASET AUDIT")
print("=" * 72)

for partition, specification in partition_specifications.items():

    path = (
        NSL_PROCESSED_DIR / specification["filename"]
    )

    parquet_metadata = pyarrow.parquet.read_metadata(
        str(path)
    )

    records = parquet_metadata.num_rows
    columns = parquet_metadata.num_columns

    if records != specification["records"]:
        raise RuntimeError(
            f"{partition}: unexpected record count."
        )

    if columns != 123:
        raise RuntimeError(
            f"{partition}: expected 123 saved columns."
        )

    print(
        f"{partition:<12} "
        f"Records: {records:<8} "
        f"Columns: {columns:<4} PASS"
    )

# ------------------------------------------------------------
# 7. Verify saved class mapping and feature schema
# ------------------------------------------------------------

schema_path = (
    NSL_PROCESSED_DIR / "nslkdd_feature_schema.json"
)

mapping_path = (
    NSL_PROCESSED_DIR / "nslkdd_class_mapping.json"
)

schema = json.loads(
    schema_path.read_text(encoding="utf-8")
)

mapping = json.loads(
    mapping_path.read_text(encoding="utf-8")
)

if schema["feature_count"] != 122:
    raise RuntimeError(
        "Encoded feature count verification failed."
    )

if schema["feature_names"] != NSL_ENCODED_FEATURE_NAMES:
    raise RuntimeError(
        "Encoded feature schema verification failed."
    )

if mapping["class_order"] != NSL_CLASS_ORDER:
    raise RuntimeError(
        "Class ordering verification failed."
    )

if mapping["class_to_index"] != NSL_CLASS_TO_INDEX:
    raise RuntimeError(
        "Class-index mapping verification failed."
    )

if mapping["attack_to_family"] != NSL_ATTACK_TO_FAMILY:
    raise RuntimeError(
        "Attack-family mapping verification failed."
    )

if len(mapping["attack_to_family"]) != 40:
    raise RuntimeError(
        "Expected 40 mapped attack labels."
    )

# ------------------------------------------------------------
# 8. Calculate combined artefact fingerprint
# ------------------------------------------------------------

fingerprint_content = json.dumps(
    {
        name: record["sha256"]
        for name, record in artefact_records.items()
    },
    sort_keys=True,
    separators=(",", ":")
).encode("utf-8")

combined_fingerprint = hashlib.sha256(
    fingerprint_content
).hexdigest()

# ------------------------------------------------------------
# 9. Create reproducibility manifest
# ------------------------------------------------------------

NSL09_MANIFEST = {
    "notebook": "09_NSLKDD_Preprocessing",
    "dataset": "NSL-KDD",
    "source_files": source_records,
    "processed_artefacts": artefact_records,
    "combined_artefact_fingerprint": combined_fingerprint,
    "data_partitions": {
        "development_train": {
            "records": 107077,
            "source": "Official KDDTrain+",
            "label_count": 5
        },
        "validation": {
            "records": 18896,
            "source": "Official KDDTrain+",
            "label_count": 5
        },
        "official_test": {
            "records": 22544,
            "source": "Official KDDTest+",
            "label_count": 5
        }
    },
    "raw_feature_count": 41,
    "encoded_feature_count": 122,
    "class_order": list(NSL_CLASS_ORDER),
    "random_seed": 42,
    "validation_ratio": 0.15,
    "encoder_fit_partition": "development_train",
    "test_used_for_fitting": False,
    "numerical_scaling": False,
    "resampling": None,
    "environment": {
        "python": sys.version.split()[0],
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit_learn": sklearn.__version__,
        "pyarrow": pyarrow.__version__
    },
    "model_training_performed": False
}

manifest_path = (
    NSL_RESULTS_DIR_09 / "NSL09_manifest.json"
)

manifest_content = (
    json.dumps(
        NSL09_MANIFEST,
        indent=2,
        sort_keys=True,
        allow_nan=False
    ) + "\n"
).encode("utf-8")

manifest_status = ns09_write_once(
    manifest_path,
    manifest_content
)

# ------------------------------------------------------------
# 10. Independently verify persisted manifest
# ------------------------------------------------------------

reloaded_manifest = json.loads(
    manifest_path.read_text(encoding="utf-8")
)

if reloaded_manifest != NSL09_MANIFEST:
    raise RuntimeError(
        "Persisted manifest verification failed."
    )

for relative_path, record in reloaded_manifest[
    "processed_artefacts"
].items():

    actual_sha256 = ns09_sha256(
        PROJECT_ROOT / relative_path
    )

    if actual_sha256 != record["sha256"]:
        raise RuntimeError(
            f"Artefact fingerprint mismatch: {relative_path}"
        )

for record in reloaded_manifest["source_files"].values():

    actual_sha256 = ns09_sha256(
        record["path"]
    )

    if actual_sha256 != record["sha256"]:
        raise RuntimeError(
            "Original source fingerprint mismatch."
        )

print("\n" + "=" * 72)
print("FINAL AUDIT")
print("=" * 72)

print("Official source files verified:", len(source_records))
print("Saved artefacts verified:", len(artefact_records))
print("Dataset row counts: PASS")
print("Five-class attack mapping: PASS")
print("Encoded feature schema: PASS")
print("Encoder reproducibility: PASS")
print("Development partition separation: PASS")
print("Official test partition integrity: PASS")
print("Source file fingerprints: PASS")
print("Artefact fingerprints: PASS")
print("Persisted manifest: PASS")

# ------------------------------------------------------------
# 11. Create notebook completion record
# ------------------------------------------------------------

NSL09_COMPLETION = {
    "notebook": "09_NSLKDD_Preprocessing",
    "status": "COMPLETE",
    "dataset": "NSL-KDD",
    "source_training_records": 125973,
    "development_training_records": 107077,
    "validation_records": 18896,
    "official_test_records": 22544,
    "original_predictors": 41,
    "encoded_predictors": 122,
    "class_count": 5,
    "class_order": list(NSL_CLASS_ORDER),
    "preprocessing_directory": str(NSL_PROCESSED_DIR),
    "manifest_path": str(manifest_path),
    "manifest_sha256": ns09_sha256(manifest_path),
    "combined_artefact_fingerprint": combined_fingerprint,
    "audit_checks_passed": True,
    "model_training_performed": False,
    "next_stage": "10_NSLKDD_Random_Forest_Replication"
}

completion_path = (
    NSL_RESULTS_DIR_09 / "NSL09_NOTEBOOK_COMPLETE.json"
)

completion_content = (
    json.dumps(
        NSL09_COMPLETION,
        indent=2,
        sort_keys=True,
        allow_nan=False
    ) + "\n"
).encode("utf-8")

completion_status = ns09_write_once(
    completion_path,
    completion_content
)

# ------------------------------------------------------------
# 12. Verify completion record
# ------------------------------------------------------------

saved_completion = json.loads(
    completion_path.read_text(encoding="utf-8")
)

if saved_completion != NSL09_COMPLETION:
    raise RuntimeError(
        "Persisted completion record verification failed."
    )

if saved_completion["status"] != "COMPLETE":
    raise RuntimeError(
        "Notebook completion status is invalid."
    )

if not saved_completion["audit_checks_passed"]:
    raise RuntimeError(
        "Final audit verification unsuccessful."
    )

NSL09_NOTEBOOK_COMPLETE = True

print("\n" + "=" * 72)
print("COMPLETION RECORD")
print("=" * 72)

print("Manifest:", manifest_status)
print("Completion record:", completion_status)
print("Status:", saved_completion["status"])
print("Encoded predictors:", saved_completion["encoded_predictors"])
print(
    "Training records:",
    saved_completion["development_training_records"]
)
print(
    "Validation records:",
    saved_completion["validation_records"]
)
print(
    "Official test records:",
    saved_completion["official_test_records"]
)
print("Combined fingerprint:", combined_fingerprint)

print("\n" + "=" * 72)
print("NOTEBOOK 09: COMPLETE")
print("Model training performed: NO")
print("Next stage: 10_NSLKDD_Random_Forest_Replication")
print("=" * 72)


CELL 12 — FINAL AUDIT AND COMPLETION RECORD

ORIGINAL SOURCE FINGERPRINTS

official_training
Path: /content/drive/MyDrive/MSc_Project/data/NSL_KDD/raw/KDDTrain+.txt
SHA-256: 1b86d2f957b33082081bba410fe129b475efebcc13c9014c3f447c8271aadf95

official_test
Path: /content/drive/MyDrive/MSc_Project/data/NSL_KDD/raw/KDDTest+.txt
SHA-256: fa46b0935342616aa83b7c2578db355b6a7aaabbc492248172c7a1e8b7ab8f84

PROCESSED ARTEFACT FINGERPRINTS
nslkdd_class_mapping.json              bb151e3ebbf34714...
nslkdd_feature_schema.json             359a350a97c0468b...
nslkdd_preprocessor.joblib             f79e4fd2491e4740...
nslkdd_split_indices.npz               d107d1bd1cb2abf4...
nslkdd_test.parquet                    e4065aefd9b2cd38...
nslkdd_train.parquet                   939465b0d86a8731...
nslkdd_validation.parquet              9db468e0d4b99d51...
NSL09_attack_inventory.csv             a879c696915b32db...
NSL09_class_distributions.csv          c6a4fdf70fd246c5...
NSL09_unknown_categories.csv         